<a href="https://colab.research.google.com/github/pranatipradhanwork-alt/Tata_Steel_machine_failure_predict/blob/feature%2Fcolab-notebook/notebooks/tata_steel_machine_failure_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Failure Prediction for Manufacturing Equipment

## Project summary
This project explores whether machine operating measurements can help identify equipment failures and support proactive maintenance. The dataset is synthetic, so results are a learning prototype and would need validation on real plant data.

**GitHub repository:** https://github.com/pranatipradhanwork-alt/Tata_Steel_machine_failure_predict

## Business problem
Unexpected equipment failures can interrupt production and increase maintenance costs. We will assess whether machine readings provide useful warning signals.

## Modeling question
Can the available operational measurements predict `Machine failure`? The training data is strongly imbalanced, so we will evaluate recall, precision, F1-score, and the confusion matrix rather than relying on accuracy alone.

## Notebook sections
1. Load and understand the data
2. Data quality checks and exploratory analysis
3. Target distribution and class imbalance
4. Feature preparation and leakage checks
5. Stratified train/validation split
6. Baseline and model comparison
7. Evaluation and feature importance
8. Conclusions, limitations, and next steps

## Data loading
Upload `train.csv` and `test.csv` to this Colab session when prompted. The raw files will not be added to the public GitHub repository.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

# Select both train.csv and test.csv when the upload window opens.
uploaded = files.upload()

required_files = {"train.csv", "test.csv"}
missing_files = required_files - set(uploaded)
if missing_files:
    raise FileNotFoundError(f"Please upload both files: {sorted(missing_files)}")

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"Training data: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"Test data: {test.shape[0]:,} rows × {test.shape[1]} columns")
display(train.head())

## Dataset overview and data quality

The training file contains 136,429 rows and the test file contains 90,954 rows. The training target is `Machine failure`; it has 2,148 failures (about 1.57%) and 134,281 non-failures. The test file does not include the target, so it will be reserved for predictions after model selection.

Initial checks found no blank cells or duplicate `id` values. The data includes product category and machine readings for air temperature, process temperature, rotational speed, torque, and tool wear. We will verify these checks below.

`TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are failure-mode flags. We will investigate their meaning and exclude them from model inputs unless the project documentation establishes they are known before failure. We will also avoid using row or product identifiers as model features unless there is a clear predictive reason.

In [ ]:
TARGET = "Machine failure"

# Confirm the expected target is present only in the labeled training data.
assert TARGET in train.columns
assert TARGET not in test.columns

print("Training shape:", train.shape)
print("Test shape:", test.shape)
print("\nColumn types:")
display(train.dtypes.rename("dtype").to_frame())

quality = pd.DataFrame({
    "missing_values": train.isna().sum(),
    "unique_values": train.nunique(dropna=False),
})
print("\nTraining data quality summary:")
display(quality)
print("Duplicate rows:", train.duplicated().sum())
print("Duplicate IDs:", train["id"].duplicated().sum())

# Inspect the target distribution and class imbalance.
target_counts = train[TARGET].value_counts().sort_index()
target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": (target_counts / len(train) * 100).round(3),
})
display(target_summary)

ax = target_counts.rename(index={0: "No failure (0)", 1: "Failure (1)"}).plot(
    kind="bar", color=["#4C78A8", "#E45756"], figsize=(6, 4)
)
ax.set_title("Machine failure target distribution")
ax.set_xlabel("Target class")
ax.set_ylabel("Number of records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()